# LangChain Tools

A `Tool` in LangChain is a small, reusable function that an LLM can call when it needs to do something outside the model itself, such as:

- perform a calculation
- query a database
- fetch data from the web
- call an API
- run a local utility

The key idea is simple: the model decides when a tool is useful, and your code executes the tool with the right arguments.

### Why use tools?
Large language models are great at reasoning, but they do not inherently have live access to the outside world. Tools give the model a way to act on real data and return results to the workflow.

### Types - 2 type

- Built In
- Custom


### 2. Tool Example

In [ ]:
from langchain_core.tools import tool

@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b
 
print(add_numbers)
print(add_numbers.name)
print(add_numbers.description)
print(add_numbers.invoke({"a": 5, "b": 7}))

name='add_numbers' description='Add two numbers together.' args_schema=<class 'langchain_core.utils.pydantic.add_numbers'> func=<function add_numbers at 0x0000029BDF4CF560>
add_numbers
Add two numbers together.
12


In [ ]:
# A tool is a callable function with metadata.
# The decorator turns a Python function into a LangChain Tool.
# LangChain uses the function signature and docstring to define the tool schema.

from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the result."""
    return a * b

result = multiply.invoke({"a": 4, "b": 6})
print(result)
print(type(result))

24
<class 'int'>


In [ ]:
# LangChain Tools are useful when the LLM needs external information or an action.
# They are often combined with an agent or a reasoning loop.
# Good tool design:
# 1. do one clear task
# 2. accept simple arguments
# 3. return easy-to-use output
# 4. keep the description precise so the model knows when to call it

from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Return a short weather summary for a city. Use this when the user asks about weather in a specific location."""
    weather_map = {
        "delhi": "Sunny and warm, around 32°C.",
        "london": "Cloudy and cool, around 18°C.",
        "new york": "Windy with light rain, around 21°C.",
    }
    return weather_map.get(city.lower(), f"No weather data available for {city}.")

print(get_weather.invoke({"city": "delhi"}))

### 3.1. Built-in Tool - DuckDuckGo Search

In [9]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('Who is virat kohli? write in 30 words')

print(results)
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

2 weeks ago - Retrieved 30 December 2022. ↑ "Virat Kohli becomes fastest captain to 5000 runs in Test cricket". India Today. 22 November 2019. Archived from the original on 23 November 2019. February 15, 2026 - ↑ "Virat Kohli nominated for Arjuna Award". The Indian Express. 30 April 2013. August 14, 2026 - A spunky, chubby teenager with gelled hair shot to fame after leading India to glory in the Under-19 World Cup at Kuala Lumpur in early 2008. In an Indian team filled with saint-like icons worthy of their own hagiographies, Virat Kohli, with ... July 15, 2026 - Virat Kohli is a famous Indian cricketer known for his excellent batting skills. He was born in Delhi and started playing cricket at a young age. Over the years, he became one of the best batsmen in the world. July 28, 2026 - Virat Kohli (b. 1988) is an Indian international cricketer and a former captain of the Indian cricket team in all three international formats. Kohli plays for (and previously captained) the Royal Challeng

### 3.2. Built-in Tool - Shell Tool

In [10]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('ls')

print(results)

Executing command:
 ls
'ls' is not recognized as an internal or external command,
operable program or batch file.



c:\Users\sidha\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_community\tools\shell\tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


### 4. Custom Tools
### 4.1. Method 1 - Using @tool


In [12]:
from langchain_core.tools import tool
# add tool decorator to the function to make it a LangChain Tool

@tool
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

result = multiply.invoke({"a":3, "b":5})
print(result)

15


### 4.2. Method 2 - Using StructuredTool

In [14]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

def multiply_func(a: int, b: int) -> int:
    return a * b

multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


C:\Users\sidha\AppData\Local\Temp\ipykernel_21428\293022868.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="The first number to add")
C:\Users\sidha\AppData\Local\Temp\ipykernel_21428\293022868.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="The second number to add")


### 4.3. Method 3 - Using BaseTool Class

In [15]:
from langchain.tools import BaseTool
from typing import Type

# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

multiply_tool = MultiplyTool()
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


C:\Users\sidha\AppData\Local\Temp\ipykernel_21428\413038991.py:7: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="The first number to add")
C:\Users\sidha\AppData\Local\Temp\ipykernel_21428\413038991.py:8: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="The second number to add")


### 5. Toolkit

In LangChain, a Toolkit is a collection of related tools grouped together for a specific purpose. It helps organize multiple tools and makes it easier to provide them to an agent or chain as a single unit.

In this example, the `MathToolkit` class wraps two custom tools: `add` and `multiply`. Both tools are registered with `@tool`, and the `get_tools()` method returns the list of tools.

Why use a Toolkit?

- Group related functionality together
- Simplify tool management
- Improve modularity and reusability
- Make it easier to add or remove tools for a specific domain

This approach is useful when you want to keep domain-specific operations, such as mathematical functions, organized and ready to use in one place.


In [ ]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b
 
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

class MathToolkit:
    def get_tools(self):
        return [add, multiply]

toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)


add => Add two numbers
multiply => Multiply two numbers


### 6. Tool Binding

In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

llm = ChatOpenAI()
llm_with_tools = llm.bind_tools([multiply])
llm_with_tools.invoke('Hi how are you')
# O/p:
# AIMessage(content="Hello! I'm here and ready to help. How can I assist you today?",
#           additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 49, 'total_tokens': 67, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-ER0CRjJk4lh6cHA6qEBAQID24tslu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None},
#          id='lc_run--01a0ca88-5c80-7b93-b63a-81960ac70c43-0', 
###        tool_calls=[], invalid_tool_calls=[], -- HERE TOOL CALL IS EMPTY BCZ THIS QTN DOESN'T NEED TO CALL TOOL, IT'S JUST A GREETING
#          usage_metadata={'input_tokens': 49, 'output_tokens': 18, 'total_tokens': 67, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})
# query = HumanMessage('can you multiply 3 with 1000')

llm_with_tools.invoke('can you multiply 3 with 1000')
# O/P:
# AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 54, 'total_tokens': 72, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-ER0LSHOgYg58DUWjEbYhWHP4sCwvc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0ca90-e909-7f52-91af-b59e66886808-0', 
###         tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 
#         'id': 'call_PQ99vo4NHy2rpJtKOyadRRjU', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 54, 'output_tokens': 18, 'total_tokens': 72, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

- Here LLM doesn't run the tool - It only suggest the tool and arguments

### 7. Tool Execution

In LangChain, the model does not directly run the tool by itself. It only decides that a tool is needed and returns a tool call with arguments such as:

- tool name
- argument values
- a unique call id

Your application is responsible for executing that tool and then feeding the result back into the model.

The execution flow is usually:

1. The LLM decides a tool is useful.
2. The response contains `tool_calls`.
3. You extract the tool name and arguments.
4. You call the actual function using `tool.invoke(...)`.
5. The tool returns a result as a `ToolMessage`.
6. The result is passed back to the model for the next reasoning step.

Example:

```python
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

result = multiply.invoke({"a": 3, "b": 5})
print(result)  # 15
```


In [ ]:
result = llm_with_tools.invoke('can you multiply 3 with 1000')
tool_call = result.tool_calls[0]
tool_result = multiply.invoke(tool_call['args'])
print(tool_result)

content='3000' name='multiply' tool_call_id='call_834dlW7i2xI1bRhunmYYpAcg'


In [22]:
from langchain_core.messages import HumanMessage, ToolMessage

result = llm_with_tools.invoke('can you multiply 3 with 1000')
tool_call = result.tool_calls[0]
actual_result = multiply.invoke(tool_call['args'])

messages = [
    HumanMessage(content="can you multiply 3 with 1000"),
    result,  # AIMessage with tool_calls
    ToolMessage(
        content=str(actual_result),
        tool_call_id=tool_call['id']  # must match the earlier tool call id
    )
]

final_reply = llm_with_tools.invoke(messages)
print(final_reply)

content='The result of multiplying 3 by 1000 is 3000.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 81, 'total_tokens': 97, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-ER0m68xPvrbykySPPcLvqL9z1tEBV', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0caaa-1ee3-7651-89c5-54df8b92c60f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 81, 'output_tokens': 16, 'total_tokens': 97, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
